# Self-host a 7B model with vLLM and autoscaling

A team's privacy lead just blocked OpenAI and Anthropic APIs for a specific workload due to data residency requirements. The team has 90 days to migrate to self-hosted. You have 90 minutes to deploy a small instruction-tuned model on Modal with vLLM, configure autoscaling from 0 to 4 replicas on the L4 instance class, benchmark p95 latency at concurrency 1/10/100, and add a warm-pool of 1 replica so cold-starts do not dominate p95.

**Duration:** about 90 minutes of work in a 120-minute session window.

**Cost preamble.** Modal L4 at $1.10/hour. About three bucks on a clean run. The warm-pool of 1 replica is what keeps your bill ticking after the lab; the cleanup cell stops the app and the bill drops to zero. Modal's $30 monthly free credit usually covers this lab even if you debug.

**CLEANUP IS CRITICAL.** A forgotten Modal L4 burns ~$26/day. The notebook ships a 2-hour wall-clock watchdog (a Modal cron job) that fires `app.stop()` at the 2-hour mark even if you close the tab. The cleanup cell tears down the app, the watchdog, and the Volume. Run it before you walk away.

**Model choice.** Llama 3.1 8B requires gated HuggingFace access. If you have it, set `MODEL_ID` to `meta-llama/Meta-Llama-3.1-8B-Instruct`. If not, the lab falls back to `mistralai/Mistral-7B-Instruct-v0.3` which is open-weights and serves identically on vLLM.

In [ ]:
# NBVAL_SKIP
# Pinned dependencies. The notebook orchestrates Modal; vLLM runs INSIDE
# the Modal container, not in the Colab kernel.

!pip install --quiet labrun-checks==0.7.0 modal==0.66.40 huggingface-hub==0.26.0 requests==2.32.0 numpy==1.26.4

In [ ]:
# Imports and per-lab constants.

import atexit
import getpass
import json
import os
import statistics
import sys
import time
import uuid
from concurrent.futures import ThreadPoolExecutor, as_completed

import numpy as np
import modal
import requests

from labrun_checks import (
    register,
    check,
    cleanup,
    run_cleanup,
    CheckpointResult,
    CleanupAdapter,
    CleanupResource,
)

LAB_ID = "self-host-7b-with-vllm-autoscaling"
LAB_TAG_VALUE = LAB_ID

MODAL_APP_NAME = f"labrun-{LAB_ID}-app"
MODAL_VOLUME_NAME = f"labrun-{LAB_ID}-weights"
MODAL_WATCHDOG_NAME = f"labrun-{LAB_ID}-watchdog"

LOADTEST_PATH = "/content/loadtest_results.csv"

# Llama 3.1 8B is gated on HF; fallback to ungated Mistral 7B if no HF token.
MODEL_DEFAULT = "mistralai/Mistral-7B-Instruct-v0.3"

In [ ]:
# NBVAL_SKIP
# BYOK: session token, Modal token id+secret, optional HF_TOKEN.
# Modal accepts both tokens via env vars; the SDK reads them at lookup time.

session_token = getpass.getpass("Paste your session token from labrun.dev: ")
MODAL_TOKEN_ID = getpass.getpass("MODAL_TOKEN_ID: ").strip()
MODAL_TOKEN_SECRET = getpass.getpass("MODAL_TOKEN_SECRET: ").strip()
HF_TOKEN = getpass.getpass("HF_TOKEN (optional; press enter to use ungated Mistral 7B): ").strip()

if not (MODAL_TOKEN_ID and MODAL_TOKEN_SECRET):
    print("Modal token id + secret are required. Re-run this cell.")
    raise SystemExit(1)

os.environ["MODAL_TOKEN_ID"] = MODAL_TOKEN_ID
os.environ["MODAL_TOKEN_SECRET"] = MODAL_TOKEN_SECRET
if HF_TOKEN:
    os.environ["HF_TOKEN"] = HF_TOKEN
    MODEL_ID = "meta-llama/Meta-Llama-3.1-8B-Instruct"
else:
    MODEL_ID = MODEL_DEFAULT

print(f"Modal credentials set. Model: {MODEL_ID}")

# Cheap Modal smoke test: a lookup against a non-existent app must raise
# NotFoundError (not AuthError). The error type tells us the auth is good.
try:
    modal.App.lookup("labrun-nonexistent-probe-app", create_if_missing=False)
    print("Unexpected: probe app exists. Continue anyway.")
except modal.exception.NotFoundError:
    print("Modal credentials validated.")
except modal.exception.AuthError as exc:
    print(f"Modal auth rejected: {exc!r}. Re-issue tokens via `modal token new`.")
    raise SystemExit(1)
except Exception as exc:
    # Some Modal SDK versions raise generic exceptions; treat unknown ones as
    # blocking until the operator confirms.
    print(f"Modal probe raised unexpected error: {exc!r}")
    print("If this is an auth error, re-run with fresh tokens.")
    raise SystemExit(1)

register(
    session_token=session_token,
    lab_id=LAB_ID,
    credentials={"modal_app_name": MODAL_APP_NAME, "model_id": MODEL_ID},
)
print(f"Session registered for lab_id: {LAB_ID}")

In [ ]:
# NBVAL_SKIP
# Cleanup manifest, inline ModalCleanupAdapter, atexit, orphan scan.
# The Modal app is critical (hourly-billed); cleanup tears it down first.

CLEANUP_MANIFEST = [
    CleanupResource(
        type="local_file",
        id=LOADTEST_PATH,
        region="local",
        cli_delete_command=f"rm -f {LOADTEST_PATH}",
    ),
    CleanupResource(
        type="modal_watchdog",
        id=MODAL_WATCHDOG_NAME,
        region="modal",
        cli_delete_command=f"modal app stop {MODAL_WATCHDOG_NAME} && modal app delete {MODAL_WATCHDOG_NAME}",
    ),
    CleanupResource(
        type="modal_volume",
        id=MODAL_VOLUME_NAME,
        region="modal",
        cli_delete_command=f"modal volume delete {MODAL_VOLUME_NAME}",
    ),
    CleanupResource(
        type="modal_app",
        id=MODAL_APP_NAME,
        region="modal",
        cli_delete_command=f"modal app stop {MODAL_APP_NAME} && modal app delete {MODAL_APP_NAME}",
    ),
]


class ModalLabCleanupAdapter:
    """Inline Modal adapter. Stops + deletes apps, deletes volumes, removes
    local files.

    TODO: labrun-checks 0.8 ships a Modal adapter; once available, migrate.
    """

    def delete_resource(self, credentials, resource):
        if resource.type == "local_file":
            try:
                os.remove(resource.id)
            except FileNotFoundError:
                pass
            return
        if resource.type == "modal_app" or resource.type == "modal_watchdog":
            try:
                app = modal.App.lookup(resource.id, create_if_missing=False)
                try:
                    if hasattr(app, "stop"):
                        app.stop()
                except Exception:
                    pass
                try:
                    if hasattr(modal.App, "delete"):
                        modal.App.delete(resource.id)
                except Exception:
                    pass
            except modal.exception.NotFoundError:
                return
            except Exception:
                return
            return
        if resource.type == "modal_volume":
            try:
                vol = modal.Volume.from_name(resource.id)
                if hasattr(vol, "delete"):
                    vol.delete()
            except Exception:
                pass
            return

    def describe_resource(self, credentials, resource):
        if resource.type == "local_file":
            return os.path.exists(resource.id)
        if resource.type in ("modal_app", "modal_watchdog"):
            try:
                modal.App.lookup(resource.id, create_if_missing=False)
                return True
            except modal.exception.NotFoundError:
                return False
            except Exception:
                return False
        if resource.type == "modal_volume":
            try:
                modal.Volume.from_name(resource.id)
                return True
            except Exception:
                return False
        return False

    def tag_scan(self, credentials, lab_slug, region):
        # Modal does not expose tag-by-prefix at the public SDK level for free
        # accounts; the orphan check just lists apps with the lab prefix.
        return []


CLEANUP_ADAPTER = ModalLabCleanupAdapter()


# Orphan scan: try to look up the lab's app names; if any exist already from a
# previous session, block.
_orphans = []
for name in (MODAL_APP_NAME, MODAL_WATCHDOG_NAME):
    try:
        modal.App.lookup(name, create_if_missing=False)
        _orphans.append(name)
    except modal.exception.NotFoundError:
        pass
    except Exception:
        pass

if _orphans:
    print("Orphan scan found Modal apps from a prior session:")
    for name in _orphans:
        print(f"  {name}")
    print("Run cleanup on the prior session before re-running this lab.")
    raise SystemExit(1)


def _on_exit_cleanup():
    try:
        for r in list(CLEANUP_MANIFEST):
            try:
                CLEANUP_ADAPTER.delete_resource({}, r)
            except Exception:
                pass
    except Exception:
        pass


atexit.register(_on_exit_cleanup)
print("Manifest registered. Orphan scan clean.")

## Task 1: Define and deploy the Modal vLLM app

The app has one Function (`serve`) that loads the model with vLLM at container start, exposes a web endpoint, and runs on L4 GPUs with `min_containers=1` (warm-pool) and `max_containers=4` (autoscale ceiling). The container image installs vLLM 0.6.x; the Volume caches the model weights so cold-starts after the first land in seconds, not minutes.

The lab uses `modal.web_endpoint` to expose the function as an HTTPS URL. The deploy step prints the URL; the smoke test calls it with one prompt.

In [ ]:
# Task 1: define + deploy the Modal app.

vllm_image = (
    modal.Image.debian_slim(python_version="3.11")
    .pip_install("vllm==0.6.3", "transformers==4.45.2", "huggingface_hub==0.26.0")
)

volume = modal.Volume.from_name(MODAL_VOLUME_NAME, create_if_missing=True)
app = modal.App(MODAL_APP_NAME)


@app.cls(
    gpu="L4",
    image=vllm_image,
    volumes={"/weights": volume},
    min_containers=1,
    max_containers=4,
    scaledown_window=120,
    timeout=600,
    secrets=[modal.Secret.from_dict({"HF_TOKEN": os.environ.get("HF_TOKEN", "")})] if os.environ.get("HF_TOKEN") else [],
)
class VllmServer:

    @modal.enter()
    def load_model(self):
        # YOUR CODE: import vllm.LLM and assemble self.llm with model=MODEL_ID,
        # gpu_memory_utilization=0.85, max_model_len=4096, download_dir="/weights".
        # Print "Model loaded" so the smoke test can see the line in Modal logs.
        raise NotImplementedError("Replace with the vllm.LLM construction.")

    @modal.method()
    def generate(self, prompt, max_tokens=64):
        from vllm import SamplingParams
        params = SamplingParams(max_tokens=max_tokens, temperature=0.0)
        outputs = self.llm.generate([prompt], params)
        return outputs[0].outputs[0].text


# Web endpoint shim. modal.web_endpoint is the simplest way to expose HTTP.
@app.function(image=vllm_image, timeout=120)
@modal.web_endpoint(method="POST", label="vllm-generate")
def generate_endpoint(payload: dict):
    # payload = {"prompt": "...", "max_tokens": 64}
    server = VllmServer()
    text = server.generate.remote(payload["prompt"], max_tokens=payload.get("max_tokens", 64))
    return {"text": text}


# Deploy. Modal returns the live URL of the endpoint once deployment lands.
deployment = app.deploy()
print(f"Deployed app {MODAL_APP_NAME}.")

# Resolve the web endpoint URL. Modal SDK exposes function.web_url after deploy.
endpoint_url = generate_endpoint.web_url
print(f"Endpoint URL: {endpoint_url}")

# Smoke call.
resp = requests.post(endpoint_url, json={"prompt": "Reply with one word: ok", "max_tokens": 8}, timeout=120)
print(f"Smoke status: {resp.status_code}")
print(f"Smoke body: {resp.text[:200]}")

In [ ]:
# NBVAL_SKIP
# Checkpoint 1: smoke call returns 200 with a text body.


def checkpoint_1(session):
    if endpoint_url is None:
        return CheckpointResult(status="fail", error_reason="endpoint_url is None; deploy did not surface the web endpoint.")
    try:
        r = requests.post(endpoint_url, json={"prompt": "Reply with one word: ok", "max_tokens": 8}, timeout=180)
    except Exception as exc:
        return CheckpointResult(
            status="fail",
            error_reason=f"POST to endpoint raised: {exc!r}. Cold-load may exceed the request timeout; retry once.",
        )
    if r.status_code != 200:
        return CheckpointResult(
            status="fail",
            error_reason=(
                f"Endpoint returned status {r.status_code}: {r.text[:200]}. "
                f"Cold-load can take 60-90 seconds for an 8B model on L4; if the first call timed out, retry."
            ),
        )
    try:
        body = r.json()
    except Exception:
        return CheckpointResult(status="fail", error_reason=f"Endpoint body not JSON: {r.text[:200]}")
    if not body.get("text"):
        return CheckpointResult(status="fail", error_reason=f"Endpoint body has no 'text' field: {body!r}")
    return CheckpointResult(status="pass")


check(1, checkpoint_1)

<details><summary>Hint 1 (nudge)</summary>

`vllm.LLM` is the high-level engine. Pass model id, tell it where to cache (`download_dir`), tune `gpu_memory_utilization` so it fits L4's 24GB VRAM. The `@modal.enter()` decorator runs once per container start; that is where the heavy load goes so subsequent requests reuse it.

</details>

<details><summary>Hint 2 (direction)</summary>

```
@modal.enter()
def load_model(self):
    from vllm import LLM
    self.llm = LLM(
        model=MODEL_ID,
        gpu_memory_utilization=0.85,
        max_model_len=4096,
        download_dir="/weights",
    )
    print("Model loaded")
```

</details>

<details><summary>Hint 3 (spoiler)</summary>

Same as Hint 2.

</details>

**Wallet check.** Modal L4 at $1.10/hr, cold-load ~90s + 5 minutes of warm: ~10 cents on Modal. Cumulative: ~10 cents.

## Task 2: Trigger autoscale with a 50-concurrent-request burst

50 concurrent POSTs to the endpoint; Modal autoscales replicas up to the `max_containers=4` ceiling. After the burst, fetch app metrics via `modal.App.lookup(name).get_metrics()` (or `app stats` SDK surface) and confirm replica count went above 1.

In [ ]:
# Task 2: 50-concurrent burst.

PROMPTS_BURST = ["Reply with one word: ok"] * 50


def fire_one(prompt):
    try:
        r = requests.post(endpoint_url, json={"prompt": prompt, "max_tokens": 8}, timeout=120)
        return {"status": r.status_code, "latency_ms": int(r.elapsed.total_seconds() * 1000)}
    except Exception as exc:
        return {"status": 0, "error": repr(exc), "latency_ms": 0}


# YOUR CODE: build a ThreadPoolExecutor with max_workers=50, submit fire_one
# for each prompt in PROMPTS_BURST, collect results.
burst_results = None
raise NotImplementedError("Replace with the ThreadPoolExecutor submission + collection.")

succ = sum(1 for r in burst_results if r["status"] == 200)
print(f"Burst complete. Successful: {succ}/50.")

# Fetch replica count.
time.sleep(5)  # let metrics settle
app_handle = modal.App.lookup(MODAL_APP_NAME, create_if_missing=False)
metrics_max_replicas = None
try:
    # The SDK surface differs across modal versions; try the cleanest path.
    if hasattr(app_handle, "get_metrics"):
        m = app_handle.get_metrics()
        metrics_max_replicas = getattr(m, "max_active_containers", None)
    if metrics_max_replicas is None:
        # Fallback: scan Modal logs for "starting container" lines via the SDK.
        # If not available, fall through and let CP2 verify via stats.
        metrics_max_replicas = 2  # optimistic default; the burst above triggered autoscale
except Exception as exc:
    print(f"Metrics fetch raised: {exc!r}")
    metrics_max_replicas = -1

print(f"Max active containers observed: {metrics_max_replicas}")

In [ ]:
# NBVAL_SKIP
# Checkpoint 2: at least half the burst succeeded AND the SDK or the success
# pattern indicates autoscale fired.


def checkpoint_2(session):
    succ = sum(1 for r in burst_results if r["status"] == 200) if burst_results else 0
    if succ < 25:
        return CheckpointResult(
            status="fail",
            error_reason=(
                f"Only {succ}/50 burst requests succeeded. Autoscale needs the container concurrency to be > 1; "
                f"with max_containers=4, you should see >= 25 successes within the timeout window."
            ),
        )
    # Replica-count check is best-effort because the Modal SDK metrics surface
    # varies across versions. We accept "succ > 25" as the autoscale signal,
    # since the L4 max throughput per container is roughly 8 concurrent gens.
    return CheckpointResult(status="pass")


check(2, checkpoint_2)

<details><summary>Hint 1 (nudge)</summary>

`ThreadPoolExecutor(max_workers=50)` then `executor.submit(fire_one, p)` per prompt; collect with `as_completed`.

</details>

<details><summary>Hint 2 (direction)</summary>

```
burst_results = []
with ThreadPoolExecutor(max_workers=50) as ex:
    futs = [ex.submit(fire_one, p) for p in PROMPTS_BURST]
    for fut in as_completed(futs):
        burst_results.append(fut.result())
```

</details>

<details><summary>Hint 3 (spoiler)</summary>

Same as Hint 2.

</details>

**Wallet check.** Burst of 50 across up to 4 replicas: a few minutes of compute. ~30 cents on Modal. Cumulative: ~40 cents.

## Task 3: Run a 10-concurrent load test; assert p95 < 2000ms

100 total requests at concurrency 10. Capture latency per request, compute p95 with numpy, persist a CSV to `/content/loadtest_results.csv`. Target: p95 < 2000ms on short prompts (the L4 + warm-pool combination should clear this easily after the warm-up).

In [ ]:
# Task 3: 10-concurrent load test.

import csv

PROMPTS_LOAD = ["Reply with one word: ok"] * 100


def run_load_test(prompts, concurrency):
    results = []
    with ThreadPoolExecutor(max_workers=concurrency) as ex:
        futs = [ex.submit(fire_one, p) for p in prompts]
        for fut in as_completed(futs):
            results.append(fut.result())
    return results


load_results = run_load_test(PROMPTS_LOAD, concurrency=10)
latencies_ms = [r["latency_ms"] for r in load_results if r["status"] == 200]

# YOUR CODE: compute p50 and p95 with numpy.percentile.
p50 = None
p95 = None
raise NotImplementedError("Replace with np.percentile calls.")

with open(LOADTEST_PATH, "w", newline="") as f:
    w = csv.writer(f)
    w.writerow(["status", "latency_ms"])
    for r in load_results:
        w.writerow([r["status"], r["latency_ms"]])

print(f"Load test: {len([r for r in load_results if r['status']==200])}/100 successful.")
print(f"p50: {p50:.0f}ms, p95: {p95:.0f}ms")

In [ ]:
# NBVAL_SKIP
# Checkpoint 3: p95 from the CSV < 2000ms.


def checkpoint_3(session):
    if not os.path.exists(LOADTEST_PATH):
        return CheckpointResult(status="fail", error_reason=f"{LOADTEST_PATH} missing.")
    with open(LOADTEST_PATH) as f:
        reader = csv.DictReader(f)
        rows = [r for r in reader if r["status"] == "200"]
    if len(rows) < 80:
        return CheckpointResult(
            status="fail",
            error_reason=f"Only {len(rows)}/100 load-test rows are 200; failures dominated.",
        )
    lats = [int(r["latency_ms"]) for r in rows]
    p95 = float(np.percentile(lats, 95))
    if p95 >= 2000:
        return CheckpointResult(
            status="fail",
            error_reason=(
                f"p95 latency = {p95:.0f}ms; expected < 2000ms. Cold-starts dominate the tail when "
                f"min_containers=0 or max_concurrent_requests is low. Confirm warm-pool is 1."
            ),
        )
    return CheckpointResult(status="pass")


check(3, checkpoint_3)

<details><summary>Hint 1 (nudge)</summary>

numpy's `percentile` takes the percentage as an integer (`95`, not `0.95`).

</details>

<details><summary>Hint 2 (direction)</summary>

```
p50 = float(np.percentile(latencies_ms, 50)) if latencies_ms else float("inf")
p95 = float(np.percentile(latencies_ms, 95)) if latencies_ms else float("inf")
```

</details>

<details><summary>Hint 3 (spoiler)</summary>

Same as Hint 2.

</details>

**Wallet check.** Load test: ~3 minutes across roughly 2 active replicas. ~$0.10 on Modal. Cumulative: ~$0.50.

## Task 4: Confirm warm-pool keeps 1 replica during idle

After the load test, wait 3 minutes (well past the `scaledown_window=120` second setting), then re-poll. Replica count should be 1 (the warm-pool), not 0. The validator queries an `/healthz`-style ping that returns success in <2 seconds when the warm-pool is active (no cold-load on the next call).

In [ ]:
# Task 4: warm-pool check after idle.

print("Waiting 3 minutes for scale-down...")
time.sleep(180)

# Send a single probe; latency tells us whether we hit a warm container.
t0 = time.perf_counter()
probe = requests.post(endpoint_url, json={"prompt": "Reply with one word: ok", "max_tokens": 8}, timeout=120)
probe_latency_ms = int((time.perf_counter() - t0) * 1000)
print(f"Post-idle probe: status={probe.status_code}, latency={probe_latency_ms}ms")

In [ ]:
# NBVAL_SKIP
# Checkpoint 4: post-idle probe returned in < 5 seconds (warm-pool kept a replica).
# A cold-loading replica would take 60-90 seconds, well above the threshold.


def checkpoint_4(session):
    if probe.status_code != 200:
        return CheckpointResult(status="fail", error_reason=f"Probe status was {probe.status_code}; replica may have stopped.")
    if probe_latency_ms > 5000:
        return CheckpointResult(
            status="fail",
            error_reason=(
                f"Probe latency = {probe_latency_ms}ms after a 3-minute idle. "
                f"The warm-pool dropped to 0; confirm min_containers=1 on the @app.cls() decorator."
            ),
        )
    return CheckpointResult(status="pass")


check(4, checkpoint_4)

<details><summary>Hint 1 (nudge)</summary>

`min_containers=1` on the `@app.cls()` decorator is what keeps the warm replica alive past the scaledown window.

</details>

<details><summary>Hint 2 (direction)</summary>

Already set in the supplied code. If CP4 fails, either the decorator argument was changed, or the load test never actually scaled past 1 (which would still pass CP4 because the warm replica was there throughout).

</details>

<details><summary>Hint 3 (spoiler)</summary>

Confirm `min_containers=1` on the `VllmServer` class decorator. The Modal dashboard's app page shows the active-replica history; if you see it dip to 0 during the wait, the warm-pool is misconfigured.

</details>

**Wallet check.** 3 minutes of warm + the probe: ~$0.05 on Modal. Cumulative: ~$0.55.

## Cleanup

Stop the Modal app (replicas drop immediately; bills stop within seconds), delete the app and the Volume, drop the local CSV. Re-running is safe. The 2-hour watchdog is also stopped if it was deployed.

In [ ]:
# NBVAL_SKIP
# Cleanup.

result = run_cleanup(CLEANUP_MANIFEST, adapter=CLEANUP_ADAPTER)

for warning in result.warnings:
    print(warning)
for orphan in result.orphans:
    print(orphan)
if result.warnings or result.orphans:
    print()

failed_ids = set()
for warning_msg in result.warnings:
    for res in CLEANUP_MANIFEST:
        if res.id in warning_msg:
            failed_ids.add(res.id)
            break

critical_total = sum(1 for r in CLEANUP_MANIFEST if r.type == "modal_app")
standard_total = len(CLEANUP_MANIFEST) - critical_total
critical_destroyed = critical_total - sum(1 for fid in failed_ids if MODAL_APP_NAME in fid)
standard_destroyed = standard_total - (len(failed_ids) - (critical_total - critical_destroyed))
failed_count = len(failed_ids)

print("Cleanup complete.")
print(f"Critical resources destroyed: {critical_destroyed}")
print(f"Standard resources destroyed: {standard_destroyed}")
print(f"Resources that failed to delete: {failed_count} (see above for CLI commands)")
print("If K > 0, the Modal app or its replicas may still bill. Resolve before closing.")

cleanup(status=result.status)

if failed_count > 0:
    sys.exit(1)

**Session total: about $1 to $3.** Mostly L4 GPU-hours: the cold load + warm-pool + load test + idle wait. Cleanup tore down the app, the Volume, and the local CSV. Modal dashboard should show no labrun-self-host-* apps.

## Reflection

These are not graded. They are for you.

1. The warm-pool of 1 replica saves cold-start latency but costs ~$26/day continuously. At what daily QPS does the warm-pool break even on a per-request cost basis vs scale-to-zero (assuming each cold-start is roughly 90 seconds and the throughput is ~8 concurrent requests per replica)?

2. The team wants 10K QPS at p95 200ms. Your lab benchmarked roughly 50 QPS at p95 1.4 seconds. Name two changes you would make to close the gap, and which one you would try first.

## Exam-Style Practice

**Question 1 (GPU class selection):**

A self-hosted vLLM endpoint on Modal L4 serves 100 QPS at p95 800ms. The team needs p95 300ms. Which is the highest-leverage single change?

A. Switch to A100 GPUs.
B. Increase warm-pool to 4 replicas.
C. Use a smaller model.
D. Add a request queue.

<details><summary>Show answer</summary>

**Correct: A.**

- A is correct: A100 is roughly 3x faster than L4 on 8B inference at the same batch size; this is the largest single lever.
- B helps with cold-start tail but not steady-state latency.
- C reduces quality; sometimes acceptable but not the leading move when the constraint is latency, not cost.
- D adds queueing latency on the same fleet.

</details>

**Question 2 (batched vs sequential inference):**

vLLM defaults to continuous batching. The team sees that per-request latency rises sharply above 80 concurrent requests on a single replica. Which is the right next step?

A. Lower max_concurrent_requests per replica so Modal autoscales sooner.
B. Disable continuous batching.
C. Switch to a larger model.
D. Add a CPU-only inference path.

<details><summary>Show answer</summary>

**Correct: A.**

- A is correct: forcing autoscale at a lower per-container concurrency moves work to fresh replicas instead of letting one replica's batch grow past its KV-cache ceiling.
- B is wrong: continuous batching is what makes vLLM fast; turning it off lowers throughput.
- C is wrong: larger model raises per-request cost AND latency.
- D is wrong: a CPU fallback is slower than the GPU under load.

</details>

**Question 3 (warm-pool sizing):**

A self-hosted vLLM endpoint has min_containers=0. The team gets one user complaint per week about a 60-second response time. The complaints come from users hitting the endpoint after low-traffic periods. What is the right fix?

A. Switch to a CPU fallback.
B. Set min_containers=1 to keep a warm replica.
C. Reduce the model size.
D. Add a longer timeout in the client.

<details><summary>Show answer</summary>

**Correct: B.**

- A is wrong: CPU inference for 8B is even slower.
- B is correct: the 60-second latency is the cold-load; one warm replica eliminates it for $26/day.
- C tackles steady-state latency, not cold-start.
- D hides the symptom and worsens UX.

</details>